# MSCheck tutorial

This notebook is the recommended way to explore and test MSCheck interactively, instead of editing/running executable lines inside a library module (e.g. `mscheck/bulkanalyse.py`).

It covers:

1. Analysing a single `.mzML` spectrum with `AnalyseMS`.
2. Running a small bulk analysis with `BulkAnalyser`.
3. How to run a full, production bulk analysis from the command line with `python -m mscheck`.

All example data used here ships with the repository under `tests/testdata/`, so this notebook runs end-to-end with no extra setup beyond the `MScheck` conda environment described in the [README](../README.md).

In [ ]:
import sys
from pathlib import Path

# Locate the repo root so `import mscheck` works whether Jupyter's cwd is
# the repo root or the notebooks/ folder.
repo_root = Path.cwd()
if not (repo_root / "mscheck").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

testdata_dir = repo_root / "tests" / "testdata"
testdata_dir

## 1. Analysing a single spectrum

`AnalyseMS` loads one `.mzML` file and searches it for a target compound (given as a SMILES string) plus a set of adduct ions.

In [ ]:
from mscheck import AnalyseMS

mzml_file = testdata_dir / "1AB-1001.mzML"

spectrum = AnalyseMS(str(mzml_file), mode="Positive")

target_smiles = "O=C(c1ccco1)N1CCN(C(=O)N2CCN(c3ccccc3)CC2)CC1"

analysis = spectrum.analyse(
    compoundsmiles=target_smiles,
    ionstoadd=["[H]", "[Na]", "[K]", "[NH4+]"],
    tolerance=1,
)

print("Ions matched:", analysis["ions"])
print("Max EIC signal:", analysis["max_EIC_signal"])

In [ ]:
from IPython.display import SVG

report_path = spectrum.create_report(folder=str(repo_root / "notebooks" / "reports"), compound_name="1AB-1001-demo")
print("Report written to:", report_path)
SVG(report_path)

## 2. Bulk analysis

`BulkAnalyser` drives the same per-spectrum analysis across many samples described in a CSV file, using a YAML config to point at the CSV, the folder of `.mzML` files, and an output/report folder.

For this demo we build a config from the bundled `tests/testdata/bulk-test` data and only take the first few rows so it runs quickly. See `tests/testdata/bulk-test/mscheck_config.yaml` for a full example config, including optional conversion-estimation settings.

In [ ]:
import tempfile
import pandas as pd
import yaml

bulk_test_dir = testdata_dir / "bulk-test"
datafiles_dir = bulk_test_dir / "datafiles"

# Take a couple of sample rows and fix up the ion notation to the SMILES
# form the library expects (e.g. "[H]" rather than "H").
demo_samples = pd.read_csv(bulk_test_dir / "bulk-test.csv").head(2)
demo_samples["reactant-ions-to-add"] = "[H]"

demo_dir = Path(tempfile.mkdtemp(prefix="mscheck_notebook_demo_"))
demo_csv_path = demo_dir / "demo_samples.csv"
demo_samples.to_csv(demo_csv_path, index=False)

demo_config = {
    "paths": {
        "csv_file": str(demo_csv_path),
        "data_dir": str(datafiles_dir),
        "output": {"base_dir": str(demo_dir / "reports")},
    },
    "parameters": {
        "analysis_types": ["reactant"],
        "modes": ["Positive"],
        "tolerance": 1,
    },
}
demo_config_path = demo_dir / "demo_config.yaml"
with open(demo_config_path, "w") as f:
    yaml.safe_dump(demo_config, f)

demo_config_path

In [ ]:
from mscheck.bulkanalyse import BulkAnalyser

analyzer = BulkAnalyser(str(demo_config_path))
analyzer.load_data()

results = analyzer.process_samples(
    analysis_types=["reactant"], modes=["Positive"], batch_size=2
)

results[[c for c in results.columns if "EIC-signal" in c or "ions-matched" in c]]

Per-sample reports were written under `demo_dir / "reports"`. Let's view the first one:

In [ ]:
reports = sorted((demo_dir / "reports").glob("**/*-report.svg"))
print(f"Generated {len(reports)} report(s)")
SVG(str(reports[0])) if reports else None

`BulkAnalyser` also exposes higher-level pieces you can call individually (as demonstrated above), or the full pipeline in one call via `analyzer.run_complete_workflow()`, which additionally calculates conversions (if enabled in the config), generates heatmap visualisations, and writes per-compound summary reports.

## 3. Running a full analysis from the command line

For a real dataset, create a YAML config (see `tests/testdata/bulk-test/mscheck_config.yaml` for a fully annotated example) and run the whole workflow with the `mscheck` CLI instead of editing/running lines inside a module:

```bash
python -m mscheck path/to/your_config.yaml
```

Optional flags:

- `--log-dir logs` — directory to write the timestamped log file to (default: `logs`)
- `--batch-size 20` — number of samples processed per batch (default: `20`)

This is equivalent to what the demo above does manually, but runs `analyzer.run_complete_workflow()` for the entire CSV.